# Posterior predictive checks of the tr-PL fit (NUTS, emcee, tempered SMC, AMIS)

Module `inference/ppc.py` (15/10/2026).  Input: a finished `main.ipynb` run (data, σ, NUTS and emcee samples if they
were run) and the `smc.ipynb` folder inside it (SMC and AMIS samples).  tr-PL: NN and ODE (modes trpl, joint);
ss-PL: exact model (modes sspl, joint, section 2b).  For every sampler, `N_DRAWS` equally weighted
draws θ_s give the model curves f_s = log10 τ_diff(θ_s) at the data points and replicated data
y_rep = f_s + s_pt ε.

**Two noise scales.**  σ is the error of the WHOLE curve and is never rescaled: the likelihood is −MSE/(2σ²) per
curve (each curve counts as one observation).  Replicated data have to be drawn point by point; the per-point noise
that gives exactly this curve likelihood is iid N(0, Nσ²), i.e. s_pt = σ√N for N data points — the per-point
equivalent of the curve-level σ, not a change of σ.

* `likelihood` — curve-level σ, drawn per point as s_pt = σ√N: the generative model of the posterior;
* `per point` — s_pt = σ: the stricter reading, σ as the RMS residual of every single point.

**Checks.**  Bands of f (parameter uncertainty) and y_rep (+ noise); coverage of the data by the 68 / 95 % y_rep
intervals; PIT u_i = P(y_rep,i ≤ y_obs,i) (uniform if calibrated, piled at 0.5 = predictive too wide);
discrepancies with posterior predictive p-values: T_mse = MSE(y − f) (the likelihood statistic) and T_ac1 = lag-1
autocorrelation of y − f along QFLS (tests the iid assumption).  p ≈ 0.5 consistent, p → 0 misfit or correlated
residuals, p → 1 noise model wider than the residuals.

**Rate equations.**  The NN curves of `N_ODE` draws per sampler are compared with the ODE curves of the same draws
(the synthetic data are ODE curves): is the NN posterior predictive also the ODE posterior predictive?

In [ ]:
# ── Which run ────────────────────────────────────────────────────────────────
CONFIG_PATH = None          # None = config snapshot of RUN_IN (settings.json)
RUN_IN = "results/2trap_85-15_film/2trap_85-15_film_joint"   # finished main.ipynb run (mode trpl)
SMC_DIR = "smc"             # smc.ipynb output folder inside RUN_IN (None = no SMC/AMIS)
EXTRA_SAMPLES = None        # {"ODE SMC": "<RUN_IN>/ode_vs_nn.../posterior_samples_ode_smc.csv"}: more posteriors to check
TIMING_CONFIG = None        # None = timing block of the config

# ── Draws ───────────────────────────────────────────────────────────────────
N_DRAWS = 2000              # NN draws per sampler
N_ODE = 300                 # draws per sampler re-evaluated with the rate equations (0 = skip)
N_JOBS = 16                 # ODE worker processes
SEED = 20261015
OUT_TAG = "ppc"             # output folder inside RUN_IN
CUDA_DEVICE = "0"

## 0. Imports, network, prior, data, likelihood

In [ ]:
import os, json, warnings
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning)

from inference import surrogate, likelihood, report, ppc, runs
from inference.prior import Prior
from inference.benchmark import timeit_call, format_mean_sd
surrogate.configure_tensorflow()

RUN_IN = Path(RUN_IN)
settings_in = json.loads((RUN_IN / "settings.json").read_text())
cfg = json.loads(Path(CONFIG_PATH).read_text()) if CONFIG_PATH else settings_in["config"]
MODE = settings_in["mode"]                       # 'trpl' | 'sspl' | 'joint'
HAS_TRPL, HAS_SSPL = MODE in ("trpl", "joint"), MODE in ("sspl", "joint")
tcfg = json.loads(Path(TIMING_CONFIG).read_text())["timing"] if TIMING_CONFIG else cfg["timing"]
REPEATS = int(tcfg["repeats"])
best = json.loads((RUN_IN / "6_best_fit.json").read_text())
Eg, npulse, T = cfg["physics"]["Eg_eV"], cfg["physics"]["npulse_cm3"], cfg["physics"]["T_K"]

nn_dir = Path(cfg["nn"]["dir"])
art = surrogate.load_network(nn_dir, cfg["nn"].get("stem"))
prior = Prior.from_artifacts(art, Eg=Eg, order_traps=cfg["prior"]["order_traps"],
                             photodoping_traps=cfg["prior"]["photodoping_traps"],
                             bounds_override=cfg["prior"]["bounds_override"], fixed=cfg["prior"].get("fixed", {}), T=T)
pb_in = pd.read_csv(RUN_IN / "prior_bounds.csv").set_index("parameter")
assert np.allclose(pb_in.loc[prior.names, "theta_lo"], prior.lo) and np.allclose(pb_in.loc[prior.names, "theta_hi"], prior.hi), \
    "prior differs from the prior of RUN_IN"
SIG = runs.run_sigmas(RUN_IN)                   # sigmas of the run (sampling_info.json or 6_best_fit.json)
sigma, sigma_ss = SIG["trpl"], SIG["sspl"]
log_post = log_post_ss = sur = None
N_PTS, SCALES, SS_SCALES = 0, {}, {}
if HAS_TRPL:          # tr-PL term alone (NN), the data and sigma of the run
    log_post, sur = runs.make_log_post(cfg, RUN_IN, art, prior, "trpl", sigma, None)
    N_PTS = sur.n_points
    SCALES = {"likelihood": sigma * np.sqrt(N_PTS), "per point": sigma}
if HAS_SSPL:          # ss-PL term alone (exact model, no surrogate)
    log_post_ss, _ = runs.make_log_post(cfg, RUN_IN, art, prior, "sspl", None, sigma_ss)
    N_SS = log_post_ss.sspl_term.observation.n_points
    SS_SCALES = {"likelihood": sigma_ss * np.sqrt(N_SS), "per point": sigma_ss}

theta_best = np.array(best["theta_best"], float)
truth_theta = None
if cfg.get("true_params"):
    tp = cfg["true_params"]
    truth_theta = np.atleast_2d(prior.from_physical(tp["Et_eV"], tp["Nt_cm3"], tp["tau_n_s"], tp["tau_p_s"], tp["krad_cm3s"]))[0]

OUT = RUN_IN / OUT_TAG
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
TIMINGS = []
def log_timing(t, n_evals=None, note=""):
    row = t.as_dict(); row.update(n_evals=n_evals, note=note); TIMINGS.append(row)
    print(t, f"| evals: {n_evals:,}" if n_evals else "")
report.save_json({"run_in": str(RUN_IN), "mode": MODE, "smc_dir": SMC_DIR, "network": str(nn_dir), "sigma_trpl": sigma,
                  "sigma_sspl": sigma_ss, "n_points": N_PTS, "noise_scales": SCALES, "noise_scales_sspl": SS_SCALES,
                  "n_draws": N_DRAWS, "n_ode": N_ODE, "seed": SEED}, OUT / "settings.json")
print(f"run: {RUN_IN} | mode {MODE} | network: {nn_dir.name} | tr-PL: {N_PTS} points, σ = {sigma}, s_pt = {SCALES} | "
      f"ss-PL: σ = {sigma_ss}, s_pt = {SS_SCALES}")

## 1. Posterior draws of every sampler

Equally weighted samples: NUTS and emcee of the main run, the final SMC particles and the resampled AMIS draws of
`smc.ipynb`.  `N_DRAWS` are picked without replacement.

In [ ]:
files = {"NUTS": RUN_IN / "posterior_samples_nuts.csv", "emcee": RUN_IN / "posterior_samples_emcee.csv"}
if SMC_DIR:
    files.update({"SMC": RUN_IN / SMC_DIR / "posterior_samples_smc.csv", "AMIS": RUN_IN / SMC_DIR / "posterior_samples_amis.csv"})
for k, p_ in (EXTRA_SAMPLES or {}).items():      # e.g. the posterior of the rate equations (ode_inference.ipynb)
    files[k] = Path(p_)
DRAWS = {}
for k, p in files.items():
    if p.exists():
        th = pd.read_csv(p)[prior.names].values
        DRAWS[k] = ppc.pick_draws(th, N_DRAWS, seed=SEED)
        print(f"{k:6s}: {len(th):6d} samples -> {len(DRAWS[k])} draws  ({p})")
    else:
        print(f"{k:6s}: missing ({p})")
COLORS = dict(zip(DRAWS, ["C0", "C1", "C2", "C4", "C3", "C5"]))

## 2. NN posterior predictive

NN curves of all draws (timed: one batched forward pass per sampler), then the checks for both noise scales.

In [ ]:
CURVES, RES, tab_nn, extra, ks = {}, {}, None, {}, []
if HAS_TRPL:
    CURVES = {}
    for k, th in DRAWS.items():
        CURVES[k], t_nn = timeit_call(ppc.nn_curves, log_post, th, repeats=REPEATS, label=f"NN curves ({k}, {len(th)} draws)")
        log_timing(t_nn, n_evals=len(th), note="NN forward passes")
    y_obs, qfls = CURVES[next(iter(CURVES))]["y_obs"], CURVES[next(iter(CURVES))]["qfls"]
    f_truth = ppc.nn_curves(log_post, truth_theta[None])["f"][0] if truth_theta is not None else None
    f_best = ppc.nn_curves(log_post, theta_best[None])["f"][0]

    RES, rows = {}, []
    for k, c in CURVES.items():
        for scale, s_pt in SCALES.items():
            RES[k, scale] = ppc.check(c["f"], y_obs, s_pt, seed=SEED)
            rows.append({"sampler": k, "noise": scale, **RES[k, scale]["summary"]})
            RES[k, scale]["bands"].assign(qfls=qfls).to_csv(OUT / f"2_bands_nn_{k}_{scale.replace(' ', '_')}.csv", index=False)
            RES[k, scale]["t"].to_csv(OUT / f"2_discrepancy_nn_{k}_{scale.replace(' ', '_')}.csv", index=False)
    tab_nn = pd.DataFrame(rows); tab_nn.to_csv(OUT / "2_ppc_nn_summary.csv", index=False)
    extra = {"rmse_best_fit_nn": float(np.sqrt(np.mean((f_best - y_obs) ** 2)))}
    if f_truth is not None:
        extra["rmse_truth_nn"] = float(np.sqrt(np.mean((f_truth - y_obs) ** 2)))
    print({k: round(v, 4) for k, v in extra.items()})
    tab_nn[["sampler", "noise", "s_pt", "coverage68", "coverage95", "data_inside_f95_band", "pit_frac_0.25_0.75",
            "mse_obs_median", "mse_rep_median", "p_mse", "ac1_obs_median", "ac1_rep_median", "p_ac1", "rmse_median_curve"]].round(4)

In [ ]:
if HAS_TRPL:
    # bands + residuals: one column per sampler (dark: f 95 %, light: y_rep 95 % at σ per point)
    ks = list(CURVES)
    fig, axs = plt.subplots(2, len(ks), figsize=(3.6 * len(ks), 5.6), sharex=True, gridspec_kw={"height_ratios": [2, 1]},
                            squeeze=False)
    for j, k in enumerate(ks):
        r = RES[k, "per point"]
        ppc.plot_bands(axs[0, j], qfls, y_obs, r, color=COLORS[k], label=k)
        if f_truth is not None:
            axs[0, j].semilogy(qfls, 10 ** f_truth, "--", color="c", lw=1.0, label="NN at truth")
        axs[0, j].set_title(f"{k}: p_mse = {r['summary']['p_mse']:.2f}, p_ac1 = {r['summary']['p_ac1']:.2f}", fontsize=8)
        ppc.plot_residuals(axs[1, j], qfls, y_obs, r, color=COLORS[k])
        if f_truth is not None:
            axs[1, j].plot(qfls, f_truth - r["bands"]["f_q50"], "--", color="c", lw=1.0)
        axs[1, j].set_xlabel(r"$\Delta E_\mathrm{F}$ (eV)")
    axs[0, 0].set_ylabel(r"$\tau_\mathrm{diff}$ (s)"); axs[1, 0].set_ylabel(r"data − median f (dec.)")
    axs[0, 0].legend(fontsize=6, loc="best")
    fig.suptitle(f"NN posterior predictive — {RUN_IN.name}, y_rep band: σ = {sigma:.4f} applied to every point "
                 f"(the likelihood's curve-level σ = {sigma:.4f} corresponds to {SCALES['likelihood']:.3f} per point)",
                 fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "2_ppc_nn_bands.png"); fig.savefig(FIG / "2_ppc_nn_bands.pdf"); plt.show()

    # same with the noise of the likelihood (curve-level σ drawn per point, σ√N): the generative model of the posterior
    fig, axs = plt.subplots(1, len(ks), figsize=(3.6 * len(ks), 3.2), sharey=True, squeeze=False)
    for j, k in enumerate(ks):
        ppc.plot_residuals(axs[0, j], qfls, y_obs, RES[k, "likelihood"], color=COLORS[k], label=k)
        axs[0, j].set_title(k, fontsize=8); axs[0, j].set_xlabel(r"$\Delta E_\mathrm{F}$ (eV)")
    axs[0, 0].set_ylabel("data − median f (dec.)")
    fig.suptitle(f"replicated data under the likelihood: curve-level σ = {sigma:.4f}, drawn per point as "
                 f"σ√N = {SCALES['likelihood']:.3f} dec.", fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "2_ppc_nn_likelihood_noise.png"); plt.show()

In [ ]:
if HAS_TRPL:
    # PIT histograms and discrepancy scatter (realised vs replicated), both noise scales
    fig, axs = plt.subplots(2, 3, figsize=(12, 6.4))
    for i, scale in enumerate(SCALES):
        for k in ks:
            r = RES[k, scale]
            axs[i, 0].hist(r["pit"], bins=np.linspace(0, 1, 11), histtype="step", lw=1.3, color=COLORS[k], label=k)
            t = r["t"]
            axs[i, 1].plot(t.mse_obs, t.mse_rep, ".", ms=1.5, alpha=0.4, color=COLORS[k],
                           label=f"{k}: p = {r['summary']['p_mse']:.2f}")
            axs[i, 2].plot(t.ac1_obs, t.ac1_rep, ".", ms=1.5, alpha=0.4, color=COLORS[k],
                           label=f"{k}: p = {r['summary']['p_ac1']:.2f}")
        axs[i, 0].axhline(N_PTS / 10, color="0.4", ls=":"); axs[i, 0].set_xlabel("PIT"); axs[i, 0].set_ylabel(f"{scale}: points")
        axs[i, 1].set_xscale("log"); axs[i, 1].set_yscale("log")
        for a, lab in ((axs[i, 1], "T_mse"), (axs[i, 2], "T_ac1")):
            lo, hi = min(a.get_xlim()[0], a.get_ylim()[0]), max(a.get_xlim()[1], a.get_ylim()[1])
            a.plot([lo, hi], [lo, hi], "k-", lw=0.6); a.set_xlabel(f"{lab}(y_obs, θ)"); a.set_ylabel(f"{lab}(y_rep, θ)")
            a.legend(fontsize=6, markerscale=5)
        axs[i, 0].legend(fontsize=6)
    fig.suptitle("PIT and posterior predictive p-values (top: curve-level σ of the likelihood, drawn per point as σ√N; "
                 "bottom: σ on every point)", fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "2_ppc_pit_discrepancy.png"); plt.show()

## 2b. ss-PL posterior predictive (modes sspl / joint)

The exact steady-state model (no surrogate) at the measured intensities for the same draws; the same checks with the
noise of the ss-PL likelihood (curve-level σ_ss, drawn per point as σ_ss √N) and σ_ss on every point.

In [ ]:
SS, tab_ss = {}, None
if HAS_SSPL:
    rows_ss = []
    for k, th in DRAWS.items():
        c_ss, t_ss = timeit_call(ppc.sspl_curves, log_post_ss, th, repeats=REPEATS, label=f"ss-PL model ({k}, {len(th)} draws)")
        log_timing(t_ss, n_evals=len(th), note="exact ss-PL model")
        x_ss, y_ss = c_ss["x"], c_ss["y_obs"]
        for scale, s_pt in SS_SCALES.items():
            SS[k, scale] = ppc.check(c_ss["f"], y_ss, s_pt, seed=SEED)
            rows_ss.append({"sampler": k, "noise": scale, **SS[k, scale]["summary"]})
            SS[k, scale]["bands"].assign(x=x_ss).to_csv(OUT / f"2b_bands_sspl_{k}_{scale.replace(' ', '_')}.csv", index=False)
    tab_ss = pd.DataFrame(rows_ss); tab_ss.to_csv(OUT / "2b_ppc_sspl_summary.csv", index=False)
    ks_ss = list(DRAWS)
    obs_ss = log_post_ss.sspl_term.observation
    fig, axs = plt.subplots(2, len(ks_ss), figsize=(3.6 * len(ks_ss), 5.6), sharex=True,
                            gridspec_kw={"height_ratios": [2, 1]}, squeeze=False)
    for j, k in enumerate(ks_ss):
        r = SS[k, "per point"]
        ppc.plot_bands(axs[0, j], x_ss, y_ss, r, color=COLORS[k], label=k, ylog10=False)
        axs[0, j].set_xscale("log")
        axs[0, j].set_title(f"{k}: p_mse = {r['summary']['p_mse']:.2f}, p_ac1 = {r['summary']['p_ac1']:.2f}", fontsize=8)
        ppc.plot_residuals(axs[1, j], x_ss, y_ss, r, color=COLORS[k])
        axs[1, j].set_xlabel(obs_ss.mode.split("_vs_")[1])
    axs[0, 0].set_ylabel(obs_ss.mode.split("_vs_")[0]); axs[1, 0].set_ylabel("data − median f")
    axs[0, 0].legend(fontsize=6)
    fig.suptitle(f"ss-PL posterior predictive (exact model) — y_rep band: σ_ss = {sigma_ss:.4f} on every point "
                 f"(curve-level σ_ss = {sigma_ss:.4f} corresponds to {SS_SCALES['likelihood']:.3f} per point)",
                 fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "2b_ppc_sspl_bands.png"); fig.savefig(FIG / "2b_ppc_sspl_bands.pdf"); plt.show()
    display(tab_ss[["sampler", "noise", "s_pt", "coverage68", "coverage95", "data_inside_f95_band", "mse_obs_median",
                    "mse_rep_median", "p_mse", "ac1_obs_median", "p_ac1", "rmse_median_curve"]].round(4))

## 3. Rate equations: NN vs ODE posterior predictive

`N_ODE` draws per sampler (the first ones of section 1) are solved with the rate equations (legacy solver, parallel
over CPU cores).  Draws whose ODE curve does not cover every compared data point are dropped and counted; data points that fewer
than half of the ODE curves reach (e.g. the t = 0 point of raw data, above the ODE start QFLS) are left out of this section.

In [ ]:
ODE, rows_ode = {}, []
if N_ODE and HAS_TRPL:
    for k, th in DRAWS.items():
        sub = th[:N_ODE]
        ODE[k], t_ode = timeit_call(ppc.ode_curves, sub, prior, npulse, art.constants, qfls, n_jobs=N_JOBS, T=T,
                                    repeats=1, label=f"ODE curves ({k}, {len(sub)} draws)")
        log_timing(t_ode, n_evals=len(sub), note=f"ODE solves, {N_JOBS} worker processes")
        # 23/10/2026: compare on the data points that most ODE curves reach (a point above the ODE start QFLS, e.g. the
        # t = 0 point of raw tr-PL data, is never covered; before, it made every draw "non-covering")
        cov = np.isfinite(ODE[k]).mean(axis=0) >= 0.5
        ok = np.all(np.isfinite(ODE[k][:, cov]), axis=1)
        f_nn = CURVES[k]["f"][:N_ODE][:, cov]
        o = ODE[k][ok][:, cov]
        r_ode = ppc.check(o, y_obs[cov], sigma, seed=SEED)
        r_nn = ppc.check(f_nn[ok], y_obs[cov], sigma, seed=SEED)
        diff = f_nn[ok] - o
        rows_ode.append({"sampler": k, "n_draws": len(sub), "n_ode_covering": int(ok.sum()),
                         "n_points_compared": int(cov.sum()), "n_points": int(cov.size),
                         "rms_nn_minus_ode": float(np.sqrt(np.mean(diff ** 2))),
                         "max_abs_median_curve_diff": float(np.max(np.abs(np.median(f_nn[ok], 0) - np.median(o, 0)))),
                         "p_mse_ode": r_ode["summary"]["p_mse"], "p_mse_nn_same_draws": r_nn["summary"]["p_mse"],
                         "p_ac1_ode": r_ode["summary"]["p_ac1"], "p_ac1_nn_same_draws": r_nn["summary"]["p_ac1"],
                         "data_inside_f95_ode": r_ode["summary"]["data_inside_f95_band"],
                         "data_inside_f95_nn_same_draws": r_nn["summary"]["data_inside_f95_band"],
                         "rmse_median_curve_ode": r_ode["summary"]["rmse_median_curve"]})
        r_ode["bands"].assign(qfls=qfls[cov]).to_csv(OUT / f"3_bands_ode_{k}.csv", index=False)
        pd.DataFrame(ODE[k]).to_csv(OUT / f"3_ode_curves_{k}.csv", index=False, float_format="%.6g")
        ODE[k] = (r_ode, r_nn, cov)
    tab_ode = pd.DataFrame(rows_ode); tab_ode.to_csv(OUT / "3_ppc_ode_vs_nn.csv", index=False)
    display(tab_ode.round(4))

In [ ]:
if N_ODE and HAS_TRPL:
    fig, axs = plt.subplots(2, len(ks), figsize=(3.6 * len(ks), 5.6), sharex=True, gridspec_kw={"height_ratios": [2, 1]},
                            squeeze=False)
    for j, k in enumerate(ks):
        r_ode, r_nn, cov = ODE[k]
        b_o, b_n = r_ode["bands"], r_nn["bands"]
        qfls_c, y_c = qfls[cov], y_obs[cov]
        a = axs[0, j]
        a.fill_between(qfls_c, 10 ** b_n["f_q2.5"], 10 ** b_n["f_q97.5"], color=COLORS[k], alpha=0.35, lw=0, label="NN f 95 %")
        a.plot(qfls_c, 10 ** b_o["f_q2.5"], "k-", lw=0.7, label="ODE f 95 %"); a.plot(qfls_c, 10 ** b_o["f_q97.5"], "k-", lw=0.7)
        a.semilogy(qfls_c, 10 ** y_c, "o", ms=2.5, color="k")
        a.set_title(f"{k}: {r_ode['summary']['n_draws']} ODE draws", fontsize=8)
        m = b_o["f_q50"].values
        axs[1, j].fill_between(qfls_c, b_n["f_q2.5"] - m, b_n["f_q97.5"] - m, color=COLORS[k], alpha=0.35, lw=0)
        axs[1, j].plot(qfls_c, b_o["f_q2.5"] - m, "k-", lw=0.7); axs[1, j].plot(qfls_c, b_o["f_q97.5"] - m, "k-", lw=0.7)
        axs[1, j].plot(qfls_c, y_c - m, "o", ms=2.5, color="k"); axs[1, j].axhline(0, color="0.3", lw=0.6)
        axs[1, j].set_xlabel(r"$\Delta E_\mathrm{F}$ (eV)")
    axs[0, 0].set_ylabel(r"$\tau_\mathrm{diff}$ (s)"); axs[1, 0].set_ylabel("− median ODE f (dec.)")
    axs[0, 0].legend(fontsize=6)
    fig.suptitle("posterior predictive: NN (band) vs rate equations (lines), same draws", fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "3_ppc_nn_vs_ode.png"); fig.savefig(FIG / "3_ppc_nn_vs_ode.pdf"); plt.show()

## 4. Timing and summary

In [ ]:
timing = pd.DataFrame(TIMINGS)
timing["time"] = [format_mean_sd(m, s, int(n), "s") for m, s, n in zip(timing.mean_s, timing.std_s, timing.repeats)]
timing.to_csv(OUT / "4_timing.csv", index=False)
summary = {"run_in": str(RUN_IN), "mode": MODE, "sigma_trpl": sigma, "sigma_sspl": sigma_ss, "n_points": N_PTS,
           "noise_scales": SCALES, "noise_scales_sspl": SS_SCALES, **extra,
           "nn": None if tab_nn is None else tab_nn.to_dict("records"), "ode_vs_nn": rows_ode,
           "sspl": None if tab_ss is None else tab_ss.to_dict("records")}
report.save_json(summary, OUT / "summary.json")
print("written:", OUT)
display(timing[["label", "time", "n_evals", "note"]])